<a href="https://colab.research.google.com/github/edwardsnj/glygen-colab-notebooks/blob/main/variants/main1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
#
# Import the modules from the github repository
#
import httpimport

with httpimport.github_repo("edwardsnj","glygen-colab-notebooks", ref="main"):
  from glygen import GlyGenDownloader
  from variants import extract_datasets, make_plotdata, map_datasets, run_binomial_test


In [7]:
#
# Make lists of GlyGen reviewed data-files for (human) glycosites
# using the GlyGenDownloader class
#

ggdl = GlyGenDownloader()

# UniProt is a mix of experimental and predicted
template = "{species}_proteoform_glycosylation_sites_uniprotkb.csv"
uniprotkb_site_files = ggdl.filenames(template,species="human")

# Some data-files have predicted sites
template = "{species}_proteoform_glycosylation_sites_predicted_*.csv"
pred_site_files = ggdl.filenames(template,species="human")

# The rest (not uniprot, not predicted) are experimental sites
template = "{species}_proteoform_glycosylation_sites_*.csv"
site_files = ggdl.filenames(template,species="human")

exp_site_files = sorted(set(site_files)-set(pred_site_files)-set(uniprotkb_site_files))


In [8]:
#
# Use the GlyGenDownloader to construct data-frames for each of these classes
# of GlyGen reviewed data-files. Set the predicted column appropraitely.
#

params = {
  "usecols": ["uniprotkb_canonical_ac","start_pos","start_aa","glycosylation_type"],
  "notna": ["uniprotkb_canonical_ac","start_pos"],
  "asint": ["start_pos"],
  "dropdups": True,
  "setcolumn": {"predicted": False}
}
glyco_site_exp = ggdl.dataframe(exp_site_files,**params)

params = {
  "usecols": ["uniprotkb_canonical_ac","start_pos","start_aa","glycosylation_type"],
  "notna": ["uniprotkb_canonical_ac","start_pos"],
  "asint": ["start_pos"],
  "dropdups": True,
  "setcolumn": {"predicted": True}
}
glyco_site_pred = ggdl.dataframe(pred_site_files,**params)

params = {
  "usecols": ["uniprotkb_canonical_ac","start_pos","start_aa",
              "glycosylation_type","xref_key"],
  "notna": ["uniprotkb_canonical_ac","start_pos"],
  "asint": ["start_pos"],
  "dropdups": True,
  "transform": {"predicted":
                lambda df: df["xref_key"].isin(["protein_xref_pubmed",
                                                "protein_xref_doi"])
                },
  "dropcols": ["xref_key"]
}
glyco_site_uniprotkb = ggdl.dataframe(uniprotkb_site_files,**params)


Using cached human_proteoform_glycosylation_sites_c_man.csv (46.86 KB).
Using cached human_proteoform_glycosylation_sites_carbbank.csv (839.00 B).
Using cached human_proteoform_glycosylation_sites_diabetes_glycomic.csv (604.89 KB).
Using cached human_proteoform_glycosylation_sites_embl.csv (1.60 MB).
Using cached human_proteoform_glycosylation_sites_glycomeatlas.csv (82.51 KB).
Using cached human_proteoform_glycosylation_sites_glyconnect.csv (11.12 MB).
Using cached human_proteoform_glycosylation_sites_gptwiki.csv (384.62 KB).
Using cached human_proteoform_glycosylation_sites_harvard.csv (149.59 KB).
Using cached human_proteoform_glycosylation_sites_literature.csv (1.89 MB).
Using cached human_proteoform_glycosylation_sites_literature_mining.csv (220.01 KB).
Using cached human_proteoform_glycosylation_sites_literature_mining_manually_verified.csv (26.10 KB).
Using cached human_proteoform_glycosylation_sites_o_gluc.csv (20.45 KB).
Using cached human_proteoform_glycosylation_sites_o_gluc

In [9]:
#
# Concatenate the three data-frames, and subset
# on the predicted column, and then remove the
# predicted column
#

import pandas as pd

glyco_sites = pd.concat([glyco_site_exp,
                         glyco_site_pred,
                         glyco_site_uniprotkb],
                        ignore_index=True)
glyco_sites.drop_duplicates(inplace=True)

glyco_site_exp = glyco_sites[~glyco_sites['predicted']]
glyco_site_exp = glyco_site_exp.drop(columns=['predicted'])

glyco_site_pred = glyco_sites[glyco_sites['predicted']]
glyco_site_pred = glyco_site_pred.drop(columns=['predicted'])

print("Experimental Glycosites:\n")
print(glyco_site_exp)
print()

print("Predicted Glycosites:\n")
print(glyco_site_pred)
print()


Experimental Glycosites:

      uniprotkb_canonical_ac glycosylation_type  start_pos start_aa
0                   P07996-1           C-linked        441      Trp
1                   P10153-1           C-linked         34      Trp
2                   P98088-1           C-linked       2122      Trp
3                   Q9UNA0-1           C-linked        573      Trp
4                   P10643-1           C-linked        503      Trp
...                      ...                ...        ...      ...
63603               P12259-1           N-linked        741      Asn
63604               P14207-1           N-linked        115      Asn
63606               Q92954-1           O-linked        604      Thr
63610               P11678-1           N-linked        113      Asn
63612               P10163-1           N-linked        213      Asn

[46110 rows x 4 columns]

Predicted Glycosites:

      uniprotkb_canonical_ac glycosylation_type  start_pos start_aa
34591               B4DH59-1           O